# 电影推荐系统 - 三种机器学习方法

本notebook展示了使用三种不同机器学习方法构建的电影推荐系统：

1. **基于内容的推荐 (Content-Based Filtering)** - 使用TF-IDF和余弦相似度
2. **基于元数据的推荐 (Metadata-Based)** - 使用类型、关键词、演员、导演
3. **协同过滤推荐 (Collaborative Filtering)** - 基于用户-电影交互矩阵

---

## 1. 导入依赖和加载数据

In [ ]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

In [ ]:
# 加载数据
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

print(f"电影数据集: {movies.shape}")
print(f"演职人员数据集: {credits.shape}")
movies.head()

## 2. 数据预处理

In [ ]:
# 辅助函数
def safe_literal_eval(x):
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

def get_list(x):
    if isinstance(x, list):
        return [i['name'] for i in x][:5]
    return []

def get_director(x):
    if isinstance(x, list):
        for crew_member in x:
            if crew_member.get('job') == 'Director':
                return crew_member.get('name', '')
    return ''

def get_top_cast(x, n=5):
    if isinstance(x, list):
        return [i['name'] for i in x][:n]
    return []

def clean_text(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    elif isinstance(x, str):
        return str.lower(x.replace(" ", ""))
    return ''

In [ ]:
# 合并数据集
credits.columns = ['id', 'title', 'cast', 'crew']
movies = movies.merge(credits[['id', 'cast', 'crew']], on='id')

# 解析JSON字段
for col in ['genres', 'keywords', 'cast', 'crew']:
    movies[col] = movies[col].apply(safe_literal_eval)

# 提取特征
movies['genres_list'] = movies['genres'].apply(get_list)
movies['keywords_list'] = movies['keywords'].apply(get_list)
movies['cast_list'] = movies['cast'].apply(get_top_cast)
movies['director'] = movies['crew'].apply(get_director)

# 清理文本
movies['genres_clean'] = movies['genres_list'].apply(clean_text)
movies['keywords_clean'] = movies['keywords_list'].apply(clean_text)
movies['cast_clean'] = movies['cast_list'].apply(clean_text)
movies['director_clean'] = movies['director'].apply(clean_text)

# 处理缺失值
movies['overview'] = movies['overview'].fillna('')

# 创建索引映射
movies = movies.reset_index(drop=True)
title_to_idx = pd.Series(movies.index, index=movies['title']).to_dict()

print(f"预处理完成！共 {len(movies)} 部电影")
movies[['title', 'genres_list', 'director', 'cast_list', 'vote_average']].head(10)

---

## 方法1: 基于内容的推荐 (Content-Based Filtering)

### 原理说明
- 使用 **TF-IDF (Term Frequency-Inverse Document Frequency)** 将电影简介转换为数值向量
- TF-IDF能够捕捉文本中重要词汇的权重
- 使用 **余弦相似度** 计算电影之间的相似性

### 优点
- 不需要用户历史数据
- 能够推荐新电影
- 可解释性强

### 缺点
- 只能推荐与用户已看过的电影相似的内容
- 无法发现用户可能喜欢的新类型

In [ ]:
# 方法1: TF-IDF + 余弦相似度
print("构建基于内容的推荐模型...")

# TF-IDF向量化
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2)  # 使用unigrams和bigrams
)

tfidf_matrix = tfidf.fit_transform(movies['overview'])

print(f"TF-IDF矩阵形状: {tfidf_matrix.shape}")
print(f"特征数量: {len(tfidf.get_feature_names_out())}")

# 计算余弦相似度
content_similarity = linear_kernel(tfidf_matrix, tfidf_matrix)

print(f"相似度矩阵形状: {content_similarity.shape}")
print("基于内容的推荐模型构建完成！")

In [ ]:
def content_based_recommend(title, similarity_matrix, top_n=10):
    """基于内容推荐电影"""
    if title not in title_to_idx:
        print(f"找不到电影: {title}")
        matches = [t for t in title_to_idx.keys() if title.lower() in t.lower()]
        if matches:
            print(f"您是否在找: {matches[:5]}")
        return None
    
    idx = title_to_idx[title]
    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n + 1]
    
    movie_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    recommendations = movies.iloc[movie_indices][['title', 'genres_list', 'vote_average', 'overview']].copy()
    recommendations['similarity_score'] = scores
    
    return recommendations

In [ ]:
# 测试方法1
print("="*60)
print("方法1测试: 基于内容的推荐 (TF-IDF)")
print("="*60)

test_movie = "The Dark Knight"
print(f"\n为《{test_movie}》推荐相似电影:\n")

recommendations = content_based_recommend(test_movie, content_similarity, top_n=10)
recommendations

---

## 方法2: 基于元数据的推荐 (Metadata-Based)

### 原理说明
- 综合考虑电影的多个元数据特征：**类型、关键词、演员、导演**
- 将所有特征组合成一个"特征汤"(soup)
- 使用 **CountVectorizer** 创建词袋模型
- 通过权重调整突出重要特征（如导演、类型）

### 优点
- 结合多维度信息
- 能够发现基于同一导演/演员的电影
- 比纯文本更准确

### 缺点
- 需要丰富的元数据
- 权重设置需要调优

In [ ]:
# 方法2: 基于元数据的推荐
print("构建基于元数据的推荐模型...")

def create_soup(row):
    """创建综合特征字符串"""
    features = []
    # 类型（权重较高，重复3次）
    features.extend(row['genres_clean'] * 3)
    # 导演（权重较高，重复3次）
    if row['director_clean']:
        features.extend([row['director_clean']] * 3)
    # 演员（权重中等，重复2次）
    features.extend(row['cast_clean'] * 2)
    # 关键词
    features.extend(row['keywords_clean'])
    return ' '.join(features)

movies['soup'] = movies.apply(create_soup, axis=1)

# 使用CountVectorizer
count_vectorizer = CountVectorizer(stop_words='english', max_features=10000)
count_matrix = count_vectorizer.fit_transform(movies['soup'])

print(f"特征矩阵形状: {count_matrix.shape}")

# 计算余弦相似度
metadata_similarity = cosine_similarity(count_matrix, count_matrix)

print("基于元数据的推荐模型构建完成！")

In [ ]:
def metadata_based_recommend(title, similarity_matrix, top_n=10):
    """基于元数据推荐电影"""
    if title not in title_to_idx:
        print(f"找不到电影: {title}")
        return None
    
    idx = title_to_idx[title]
    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n + 1]
    
    movie_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    recommendations = movies.iloc[movie_indices][['title', 'genres_list', 'director', 'cast_list', 'vote_average']].copy()
    recommendations['similarity_score'] = scores
    
    return recommendations

In [ ]:
# 测试方法2
print("="*60)
print("方法2测试: 基于元数据的推荐")
print("="*60)

test_movie = "The Dark Knight"
print(f"\n为《{test_movie}》推荐相似电影:\n")

recommendations = metadata_based_recommend(test_movie, metadata_similarity, top_n=10)
recommendations

---

## 方法3: 协同过滤推荐 (Collaborative Filtering)

### 原理说明
协同过滤是推荐系统中最经典的方法，核心思想是：
- **基于用户的CF (User-Based)**: 找到与目标用户相似的用户，推荐他们喜欢的电影
- **基于物品的CF (Item-Based)**: 找到与目标电影相似的电影（基于用户行为）
- **矩阵分解 (SVD)**: 将用户-物品矩阵分解为潜在因子，捕捉隐藏模式

### 关键概念
由于TMDB数据集没有用户评分数据，我们模拟生成用户-电影交互矩阵来演示协同过滤。

### 优点
- 能够发现隐藏的用户偏好模式
- 可以推荐与用户历史不相似但可能喜欢的电影
- SVD可以处理稀疏数据

### 缺点
- 需要大量用户行为数据
- 冷启动问题（新用户/新电影）
- 计算复杂度较高

In [ ]:
# 方法3: 协同过滤
print("构建协同过滤推荐模型...")

# 生成模拟的用户-电影评分矩阵
def generate_user_movie_matrix(n_users=1000, sparsity=0.02):
    """
    生成模拟的用户-电影评分矩阵
    基于电影特征模拟用户偏好
    """
    n_movies = len(movies)
    ratings = np.zeros((n_users, n_movies))
    
    # 获取所有类型
    all_genres = set()
    for genres in movies['genres_list']:
        all_genres.update(genres)
    genre_list = sorted(list(all_genres))
    
    np.random.seed(42)
    
    for user_id in range(n_users):
        # 用户偏好的类型
        n_preferred = np.random.randint(1, 4)
        preferred_genres = np.random.choice(genre_list, size=min(n_preferred, len(genre_list)), replace=False)
        
        # 用户观看的电影数量
        n_watched = int(n_movies * sparsity * np.random.uniform(0.5, 2.0))
        n_watched = max(10, min(n_watched, n_movies // 10))
        
        # 选择电影概率
        movie_probs = np.ones(n_movies) * 0.1
        
        for movie_idx in range(n_movies):
            movie_genres = movies.iloc[movie_idx]['genres_list']
            if any(g in preferred_genres for g in movie_genres):
                movie_probs[movie_idx] = 0.8
            vote_avg = movies.iloc[movie_idx]['vote_average']
            movie_probs[movie_idx] *= (vote_avg / 10.0 + 0.5)
        
        movie_probs = movie_probs / movie_probs.sum()
        watched_movies = np.random.choice(n_movies, size=n_watched, replace=False, p=movie_probs)
        
        # 生成评分
        for movie_idx in watched_movies:
            movie_genres = movies.iloc[movie_idx]['genres_list']
            vote_avg = movies.iloc[movie_idx]['vote_average']
            base_rating = 2.5 + (vote_avg - 5) / 2
            
            if any(g in preferred_genres for g in movie_genres):
                base_rating += np.random.uniform(0.5, 1.5)
            else:
                base_rating += np.random.uniform(-1.0, 0.5)
            
            rating = np.clip(base_rating + np.random.normal(0, 0.5), 1, 5)
            ratings[user_id, movie_idx] = round(rating, 1)
    
    return ratings

# 生成用户-电影矩阵
user_movie_matrix = generate_user_movie_matrix(n_users=1000)

n_ratings = np.count_nonzero(user_movie_matrix)
sparsity = 1 - (n_ratings / (user_movie_matrix.shape[0] * user_movie_matrix.shape[1]))

print(f"用户-电影矩阵形状: {user_movie_matrix.shape}")
print(f"评分数量: {n_ratings}")
print(f"矩阵稀疏度: {sparsity:.2%}")

In [ ]:
# 可视化用户-电影评分矩阵
plt.figure(figsize=(14, 8))
subset = user_movie_matrix[:50, :100]
sns.heatmap(subset, cmap='YlOrRd', xticklabels=False, yticklabels=False)
plt.title('User-Movie Rating Matrix (Sample: 50 users x 100 movies)', fontsize=14)
plt.xlabel('Movies')
plt.ylabel('Users')
plt.tight_layout()
plt.show()

### 3.1 基于物品的协同过滤 (Item-Based CF)

In [ ]:
# 基于物品的协同过滤
print("构建基于物品的协同过滤...")

# 转置矩阵，使电影在行上
movie_user_matrix = user_movie_matrix.T

# 计算物品相似度矩阵
item_similarity_cf = cosine_similarity(movie_user_matrix)

print(f"物品相似度矩阵形状: {item_similarity_cf.shape}")

def item_based_cf_recommend(title, top_n=10):
    """基于物品的协同过滤推荐"""
    if title not in title_to_idx:
        print(f"找不到电影: {title}")
        return None
    
    movie_idx = title_to_idx[title]
    sim_scores = list(enumerate(item_similarity_cf[movie_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n + 1]
    
    movie_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    recommendations = movies.iloc[movie_indices][['title', 'genres_list', 'vote_average', 'popularity']].copy()
    recommendations['cf_similarity'] = scores
    
    return recommendations

print("基于物品的协同过滤构建完成！")

### 3.2 基于用户的协同过滤 (User-Based CF)

In [ ]:
# 基于用户的协同过滤
print("构建基于用户的协同过滤...")

# 计算用户相似度矩阵
user_similarity_cf = cosine_similarity(user_movie_matrix)

print(f"用户相似度矩阵形状: {user_similarity_cf.shape}")

def user_based_cf_recommend(title, top_n=10):
    """基于用户的协同过滤推荐"""
    if title not in title_to_idx:
        print(f"找不到电影: {title}")
        return None
    
    movie_idx = title_to_idx[title]
    
    # 找到喜欢这部电影的用户
    movie_ratings = user_movie_matrix[:, movie_idx]
    liked_users = np.where(movie_ratings >= 4.0)[0]
    
    if len(liked_users) == 0:
        liked_users = np.where(movie_ratings > 0)[0][:10]
    
    if len(liked_users) == 0:
        return item_based_cf_recommend(title, top_n)
    
    # 聚合这些用户的偏好
    user_preferences = np.zeros(len(movies))
    
    for user_id in liked_users:
        similar_users = np.argsort(user_similarity_cf[user_id])[::-1][1:11]
        for sim_user in similar_users:
            sim_score = user_similarity_cf[user_id, sim_user]
            user_preferences += sim_score * user_movie_matrix[sim_user]
    
    user_preferences[movie_idx] = -1
    top_indices = np.argsort(user_preferences)[::-1][:top_n]
    scores = user_preferences[top_indices]
    
    if scores.max() > 0:
        scores = scores / scores.max()
    
    recommendations = movies.iloc[top_indices][['title', 'genres_list', 'vote_average', 'popularity']].copy()
    recommendations['cf_score'] = scores
    
    return recommendations

print("基于用户的协同过滤构建完成！")

### 3.3 基于SVD矩阵分解的协同过滤

In [ ]:
# 基于SVD的协同过滤
print("构建基于SVD的协同过滤...")

# 中心化评分矩阵
user_ratings_mean = np.mean(user_movie_matrix, axis=1)
ratings_centered = user_movie_matrix - user_ratings_mean.reshape(-1, 1)

# SVD分解
n_factors = 50
U, sigma, Vt = svds(csr_matrix(ratings_centered), k=n_factors)
sigma = np.diag(sigma)

print(f"SVD分解 - 用户因子: {U.shape}, 物品因子: {Vt.shape}")
print(f"潜在因子数量: {n_factors}")

# 预测评分矩阵
svd_predictions = np.dot(np.dot(U, sigma), Vt) + user_ratings_mean.reshape(-1, 1)

def svd_cf_recommend(title, top_n=10):
    """基于SVD的协同过滤推荐"""
    if title not in title_to_idx:
        print(f"找不到电影: {title}")
        return None
    
    movie_idx = title_to_idx[title]
    
    # 使用物品因子计算相似度
    item_factors = Vt.T  # (n_movies, n_factors)
    target_factor = item_factors[movie_idx].reshape(1, -1)
    similarities = cosine_similarity(target_factor, item_factors)[0]
    
    sim_scores = list(enumerate(similarities))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n + 1]
    
    movie_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    recommendations = movies.iloc[movie_indices][['title', 'genres_list', 'vote_average', 'popularity']].copy()
    recommendations['svd_similarity'] = scores
    
    return recommendations

print("基于SVD的协同过滤构建完成！")

In [ ]:
# 测试协同过滤方法
print("="*60)
print("方法3测试: 协同过滤推荐")
print("="*60)

test_movie = "The Dark Knight"
print(f"\n目标电影: 《{test_movie}》")

print("\n--- Item-Based CF ---")
display(item_based_cf_recommend(test_movie, top_n=5))

print("\n--- User-Based CF ---")
display(user_based_cf_recommend(test_movie, top_n=5))

print("\n--- SVD-Based CF ---")
display(svd_cf_recommend(test_movie, top_n=5))

---

## 混合推荐系统

结合三种方法的优势，使用加权平均来生成最终推荐

In [ ]:
def hybrid_recommend(title, weights=(0.3, 0.4, 0.3), top_n=10):
    """
    混合推荐
    weights: (内容权重, 元数据权重, 协同过滤权重)
    """
    if title not in title_to_idx:
        print(f"找不到电影: {title}")
        return None
    
    idx = title_to_idx[title]
    n_movies = len(movies)
    
    # 计算综合分数
    hybrid_scores = np.zeros(n_movies)
    
    # 内容相似度
    hybrid_scores += weights[0] * content_similarity[idx]
    
    # 元数据相似度
    hybrid_scores += weights[1] * metadata_similarity[idx]
    
    # 协同过滤相似度
    hybrid_scores += weights[2] * item_similarity_cf[idx]
    
    # 排序
    sim_scores = list(enumerate(hybrid_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n + 1]
    
    movie_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    recommendations = movies.iloc[movie_indices][['title', 'genres_list', 'director', 'vote_average', 'popularity']].copy()
    recommendations['hybrid_score'] = scores
    
    return recommendations

In [ ]:
# 测试混合推荐
print("="*60)
print("混合推荐测试")
print("="*60)

test_movie = "Inception"
print(f"\n为《{test_movie}》生成混合推荐:\n")

recommendations = hybrid_recommend(test_movie, weights=(0.3, 0.4, 0.3), top_n=10)
recommendations

---

## 三种方法对比

In [ ]:
def compare_methods(title, top_n=5):
    """比较三种推荐方法"""
    print("="*70)
    print(f"三种推荐方法对比 - 目标电影: {title}")
    print("="*70)
    
    if title in title_to_idx:
        idx = title_to_idx[title]
        movie_info = movies.iloc[idx]
        print(f"\n电影信息:")
        print(f"  类型: {movie_info['genres_list']}")
        print(f"  导演: {movie_info['director']}")
        print(f"  演员: {movie_info['cast_list'][:3]}")
        print(f"  评分: {movie_info['vote_average']}")
    
    print("\n" + "-"*70)
    print("方法1: 基于内容 (TF-IDF)")
    print("-"*70)
    rec1 = content_based_recommend(title, content_similarity, top_n)
    if rec1 is not None:
        for _, row in rec1.iterrows():
            print(f"  {row['title']:<40} 相似度: {row['similarity_score']:.3f}  评分: {row['vote_average']}")
    
    print("\n" + "-"*70)
    print("方法2: 基于元数据 (类型+导演+演员)")
    print("-"*70)
    rec2 = metadata_based_recommend(title, metadata_similarity, top_n)
    if rec2 is not None:
        for _, row in rec2.iterrows():
            print(f"  {row['title']:<40} 相似度: {row['similarity_score']:.3f}  导演: {row['director']}")
    
    print("\n" + "-"*70)
    print("方法3: 协同过滤 (Item-Based CF)")
    print("-"*70)
    rec3 = item_based_cf_recommend(title, top_n)
    if rec3 is not None:
        for _, row in rec3.iterrows():
            print(f"  {row['title']:<40} CF相似度: {row['cf_similarity']:.3f}  评分: {row['vote_average']}")
    
    print("\n" + "-"*70)
    print("混合推荐 (综合三种方法)")
    print("-"*70)
    rec4 = hybrid_recommend(title, top_n=top_n)
    if rec4 is not None:
        for _, row in rec4.iterrows():
            print(f"  {row['title']:<40} 综合分数: {row['hybrid_score']:.3f}  评分: {row['vote_average']}")

In [ ]:
# 测试多部电影
test_movies = ["Avatar", "The Dark Knight", "Interstellar", "Titanic"]

for movie in test_movies:
    compare_methods(movie, top_n=5)
    print("\n")

---

## 为用户推荐电影

In [ ]:
def recommend_for_user(user_id, top_n=10):
    """为特定用户推荐电影"""
    if user_id >= user_movie_matrix.shape[0]:
        print(f"用户ID超出范围")
        return None
    
    # 获取用户已看过的电影
    watched = np.where(user_movie_matrix[user_id] > 0)[0]
    
    # 使用SVD预测评分
    predictions = svd_predictions[user_id].copy()
    predictions[watched] = -1  # 排除已看过的
    
    top_indices = np.argsort(predictions)[::-1][:top_n]
    scores = predictions[top_indices]
    
    recommendations = movies.iloc[top_indices][['title', 'genres_list', 'vote_average', 'popularity']].copy()
    recommendations['predicted_score'] = scores
    
    # 显示用户已看过的高分电影
    watched_high_rated = [(i, user_movie_matrix[user_id, i]) for i in watched if user_movie_matrix[user_id, i] >= 4]
    watched_high_rated.sort(key=lambda x: x[1], reverse=True)
    
    print(f"用户 {user_id} 喜欢的电影 (评分>=4):")
    for movie_idx, rating in watched_high_rated[:5]:
        print(f"  - {movies.iloc[movie_idx]['title']} (评分: {rating})")
    
    print(f"\n为用户 {user_id} 推荐的电影:")
    return recommendations

# 为用户0推荐电影
recommend_for_user(user_id=0, top_n=10)

---

## 可视化: 协同过滤相似度矩阵

In [ ]:
# 可视化协同过滤物品相似度
n_movies_vis = 30
subset = item_similarity_cf[:n_movies_vis, :n_movies_vis]

titles = [movies.iloc[i]['title'][:20] + '...' if len(movies.iloc[i]['title']) > 20 
          else movies.iloc[i]['title'] for i in range(n_movies_vis)]

plt.figure(figsize=(14, 12))
sns.heatmap(subset, cmap='coolwarm', xticklabels=titles, yticklabels=titles, 
            center=0, annot=False)
plt.title('Item-Based Collaborative Filtering Similarity Matrix', fontsize=14)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

---

## 交互式推荐

输入电影名称获取推荐

In [ ]:
def search_movie(query):
    """搜索电影"""
    matches = movies[movies['title'].str.contains(query, case=False, na=False)]
    return matches[['title', 'genres_list', 'vote_average', 'release_date']].head(10)

# 搜索示例
print("搜索电影:")
search_movie("Star Wars")

In [ ]:
# 获取热门电影
print("热门电影 Top 20:")
movies.nlargest(20, 'popularity')[['title', 'genres_list', 'vote_average', 'popularity']]

In [ ]:
# 自定义推荐
# 修改下面的电影名称来获取推荐
my_movie = "The Matrix"  # <-- 修改这里

compare_methods(my_movie, top_n=8)

---

## 总结

| 方法 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **基于内容** | TF-IDF + 余弦相似度 | 不需要用户数据，可解释性强 | 只能推荐相似内容 |
| **基于元数据** | 类型+导演+演员特征组合 | 多维度信息，更准确 | 需要丰富元数据 |
| **协同过滤** | 基于用户行为的相似度 | 能发现隐藏偏好，可推荐新类型 | 需要用户数据，冷启动问题 |
| **混合推荐** | 加权组合 | 结合所有方法优势 | 需要调参 |

### 协同过滤子方法

| 方法 | 原理 | 适用场景 |
|------|------|------|
| **User-Based CF** | 找相似用户推荐 | 用户数量少，行为丰富 |
| **Item-Based CF** | 找相似物品推荐 | 物品数量少，用户数量大 |
| **SVD矩阵分解** | 潜在因子模型 | 稀疏数据，需要降维 |